# Stellar Contamination Grid: ε(λ) for Two-Spots Models

This notebook computes the wavelength-dependent stellar contamination factor
$ \epsilon(\lambda) $ using POSEIDON's `stellar_contamination` for a grid of:

- Photosphere temperatures: `T_s_values`
- Spot covering fractions: `f_spot_values`
- Facula covering fractions: `f_fac_values`


In [ ]:
from __future__ import annotations

import itertools
from collections import defaultdict
from concurrent.futures import ProcessPoolExecutor
from pathlib import Path
from typing import Iterable, List, Sequence, Tuple

import matplotlib.pyplot as plt
import numpy as np
from POSEIDON.constants import R_Sun
from POSEIDON.core import create_star
from POSEIDON.stellar import stellar_contamination

In [ ]:
# ----------------------------
# Global stellar parameters
# ----------------------------
R_S: float = 0.468 * R_Sun
LOG_G_S: float = 4.6
MET_S: float = 0.0

# Photosphere temperatures (K)
T_S_VALUES: np.ndarray = np.array([3400.0, 3500.0, 3600.0], dtype=float)

# Wavelength grid (μm). This notebook uses the values in waves.txt as-is.
WL_UM: np.ndarray = np.loadtxt("waves.txt", dtype=float)

# Covering fractions (uniform grids)
F_SPOT_VALUES: np.ndarray = np.linspace(0.0, 0.3, 4)  # 0.00, 0.10, 0.20, 0.30
F_FAC_VALUES: np.ndarray = np.linspace(0.0, 0.4, 4)   # 0.00, 0.133..., 0.266..., 0.40

# Output directory for individual epsilon files
OUT_DIR = Path("TLS")
OUT_DIR.mkdir(parents=True, exist_ok=True)


## Worker definition (parallel execution)

Each worker receives a tuple `(T_s, f_spot, f_fac)` and returns:

- `T_s`, `f_spot`, `f_fac`
- `epsilon`: the contamination spectrum $ \epsilon(\lambda) $

We follow your Rackham et. al. 2017, mapping:
- $T_{\rm spot} = 0.86 \, T_s$
- $T_{\rm fac} = T_s + 100 \, \mathrm{K}$


In [ ]:
def compute_epsilon_for_case(
    args: Tuple[float, float, float]
    ) -> Tuple[float, float, float, np.ndarray]:
    """
    Compute epsilon(λ) for a single (T_s, f_spot, f_fac) case.

    Parameters
    ----------
    args
        Tuple of (T_s, f_spot, f_fac).

    Returns
    -------
    Tuple[float, float, float, np.ndarray]
        (T_s, f_spot, f_fac, epsilon_lambda).
    """
    t_s, f_spot, f_fac = args

    t_spot = 0.86 * t_s
    t_fac = t_s + 100.0

    star = create_star(
        R_S,
        t_s,
        LOG_G_S,
        MET_S,
        stellar_grid="phoenix",
        stellar_contam="two_spots",
        f_spot=f_spot,
        T_spot=t_spot,
        f_fac=f_fac,
        T_fac=t_fac,
    )

    epsilon = stellar_contamination(star, WL_UM)
    return t_s, f_spot, f_fac, np.asarray(epsilon, dtype=float)


## Task grid and parallel execution

We build all combinations of:
- `T_s_values × f_spot_values × f_fac_values`

Then we run them in parallel with `ProcessPoolExecutor`.


In [ ]:
def build_tasks(
    t_s_values: Sequence[float],
    f_spot_values: Sequence[float],
    f_fac_values: Sequence[float],
) -> List[Tuple[float, float, float]]:
    """Create the full task list (T_s, f_spot, f_fac)."""
    tasks: List[Tuple[float, float, float]] = []
    for t_s in t_s_values:
        for f_spot, f_fac in itertools.product(f_spot_values, f_fac_values):
            tasks.append((float(t_s), float(f_spot), float(f_fac)))
    return tasks


def run_parallel(tasks: Iterable[Tuple[float, float, float]]) -> List[Tuple[float, float, float, np.ndarray]]:
    """Execute the task list in parallel."""
    results: List[Tuple[float, float, float, np.ndarray]] = []
    with ProcessPoolExecutor() as executor:
        for res in executor.map(compute_epsilon_for_case, tasks):
            results.append(res)
    return results


def regroup_by_temperature(
    results: Sequence[Tuple[float, float, float, np.ndarray]]
) -> dict[float, list[tuple[float, float, np.ndarray]]]:
    """Group results by T_s."""
    by_t_s: dict[float, list[tuple[float, float, np.ndarray]]] = defaultdict(list)
    for t_s, f_spot, f_fac, epsilon in results:
        by_t_s[t_s].append((f_spot, f_fac, epsilon))
    return by_t_s


In [ ]:
def save_epsilon_curve(
    out_dir: Path,
    t_s: float,
    f_spot: float,
    f_fac: float,
    wl_um: np.ndarray,
    epsilon: np.ndarray,
) -> Path:
    """Save (wl, epsilon) into a deterministic filename."""
    filename = out_dir / f"epsilon_T{int(t_s)}_fspot{f_spot:.3f}_ffac{f_fac:.3f}.txt"
    np.savetxt(filename, np.column_stack((wl_um, epsilon)))
    return filename


def plot_epsilon_summary(
    wl_um: np.ndarray,
    by_t_s: dict[float, list[tuple[float, float, np.ndarray]]],
    ) -> None:
    """Plot faint individual curves + mean curve per T_s."""
    color_map = {
        3400.0: "tab:blue",
        3500.0: "tab:orange",
        3600.0: "tab:green",
    }

    plt.figure(figsize=(12, 8))

    for t_s in sorted(by_t_s.keys()):
        cases = by_t_s[t_s]
        color = color_map.get(t_s, "tab:gray")

        all_eps = []
        for f_spot, f_fac, epsilon in cases:
            all_eps.append(epsilon)

            # Individual curve (thin + transparent)
            plt.plot(wl_um, epsilon, color=color, alpha=0.25, linewidth=0.8)

        all_eps_arr = np.asarray(all_eps, dtype=float)
        epsilon_mean = all_eps_arr.mean(axis=0)

        plt.plot(
            wl_um,
            epsilon_mean,
            color=color,
            alpha=0.9,
            linewidth=2.0,
            label=f"T_s = {int(t_s)} K (mean)",
        )

    plt.xlabel("Wavelength (μm)")
    plt.ylabel("Stellar contamination ε(λ)")
    plt.title("Stellar contamination for multiple T_s and covering fractions")
    plt.legend(fontsize=9)
    plt.tight_layout()
    plt.show()


In [ ]:
if __name__ == "__main__":
    tasks = build_tasks(T_S_VALUES, F_SPOT_VALUES, F_FAC_VALUES)

    results = run_parallel(tasks)
    by_t_s = regroup_by_temperature(results)

    # Save all curves
    for t_s, cases in by_t_s.items():
        for f_spot, f_fac, epsilon in cases:
            save_epsilon_curve(
                out_dir=OUT_DIR,
                t_s=t_s,
                f_spot=f_spot,
                f_fac=f_fac,
                wl_um=WL_UM,
                epsilon=epsilon,
            )

    # Plot summary
    plot_epsilon_summary(WL_UM, by_t_s)
